# Introduction 
From the previous exercise and considering how sessions are defined, we would like to apply an 
ML use case to forecast a metric. You may choose one of the following: 
- Average duration of sessions 
- Number of sessions

# Assignment 
Select the top 1 user who has the highest number of sessions. Forecast the next 3 months of 
your selected metric, starting from the last available record for that user 

In [1]:
from prophet import Prophet
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

Importing plotly failed. Interactive plots will not work.


In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
DATA_FILE = "../data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv"
OUTPUT_FILE = "../data/results/exercise3_forecast.tsv"

SCHEMA = StructType([
    StructField("user_id", StringType()),
    StructField("timestamp_str", StringType()),
    StructField("artist_id", StringType()),
    StructField("artist_name", StringType()),
    StructField("track_id", StringType()),
    StructField("track_name", StringType()),
])

SESSION_GAP_IN_MIN = 20
FORECAST_METRIC = "session_count"  # or "avg_duration_minutes"
FORECAST_HORIZON_DAYS = 90

In [4]:
df = spark.read.csv(DATA_FILE, sep="\t", header=False, schema=SCHEMA)

In [5]:
df.show()

+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|    user_id|       timestamp_str|           artist_id|    artist_name|            track_id|          track_name|
+-----------+--------------------+--------------------+---------------+--------------------+--------------------+
|user_000001|2009-05-04T23:08:57Z|f1b1cf71-bd35-4e9...|      Deep Dish|                NULL|Fuck Me Im Famous...|
|user_000001|2009-05-04T13:54:10Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Composition 0919 ...|
|user_000001|2009-05-04T13:52:04Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc2 (Live_2009_4_15)|
|user_000001|2009-05-04T13:42:52Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Hibari (Live_2009...|
|user_000001|2009-05-04T13:42:11Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc1 (Live_2009_4_15)|
|user_000001|2009-05-04T13:38:31Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|To Stanford (Liv

In [6]:
df = df \
    .withColumn("started_at", F.to_timestamp("timestamp_str")) \
    .withColumn("session", F.session_window("started_at", F.lit(f"{SESSION_GAP_IN_MIN} minutes")))

df = df.withColumn(
    "session_id",
    F.concat_ws("_", F.col("user_id"), F.col("session.start").cast("string")),
)

In [7]:
df.show()

+-----------+--------------------+--------------------+---------------+--------------------+--------------------+-------------------+--------------------+--------------------+
|    user_id|       timestamp_str|           artist_id|    artist_name|            track_id|          track_name|         started_at|             session|          session_id|
+-----------+--------------------+--------------------+---------------+--------------------+--------------------+-------------------+--------------------+--------------------+
|user_000001|2009-05-04T23:08:57Z|f1b1cf71-bd35-4e9...|      Deep Dish|                NULL|Fuck Me Im Famous...|2009-05-04 23:08:57|{2009-05-04 23:08...|user_000001_2009-...|
|user_000001|2009-05-04T13:54:10Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Composition 0919 ...|2009-05-04 13:54:10|{2009-05-04 13:54...|user_000001_2009-...|
|user_000001|2009-05-04T13:52:04Z|a7f7df4a-77d8-4f1...|       坂本龍一|                NULL|Mc2 (Live_2009_4_15)|2009-05-04 13:5

In [8]:
session_summary = df.groupBy("user_id", "session_id", "session").agg(
    F.min("started_at").alias("session_start"),
    F.max("started_at").alias("session_end"),
    F.count("*").alias("track_count"),
    (
        (F.unix_timestamp(F.max("started_at")) - F.unix_timestamp(F.min("started_at"))) / 60
    ).alias("duration_minutes"),
)

In [9]:
session_summary.show()

+-----------+--------------------+--------------------+-------------------+-------------------+-----------+----------------+
|    user_id|          session_id|             session|      session_start|        session_end|track_count|duration_minutes|
+-----------+--------------------+--------------------+-------------------+-------------------+-----------+----------------+
|user_000001|user_000001_2006-...|{2006-08-13 17:02...|2006-08-13 17:02:22|2006-08-13 17:02:22|          1|             0.0|
|user_000001|user_000001_2006-...|{2006-08-13 17:23...|2006-08-13 17:23:11|2006-08-13 17:23:11|          1|             0.0|
|user_000001|user_000001_2006-...|{2006-08-13 17:56...|2006-08-13 17:56:14|2006-08-13 17:56:14|          1|             0.0|
|user_000001|user_000001_2006-...|{2006-08-15 12:57...|2006-08-15 12:57:01|2006-08-15 12:57:01|          1|             0.0|
|user_000001|user_000001_2006-...|{2006-08-16 11:04...|2006-08-16 11:04:02|2006-08-16 11:04:02|          1|             0.0|


In [10]:
top_user = (
    session_summary
    .groupBy("user_id")
    .agg(F.count("session_id").alias("num_sessions"))
    .orderBy("num_sessions", ascending=False)
    .first()["user_id"]
)

print(f"Top user by session count: {top_user}")

Top user by session count: user_000949


In [11]:
user_sessions = (
    session_summary
    .filter(F.col("user_id") == top_user)
    .withColumn("date", F.to_date("session_start"))
)

if FORECAST_METRIC == "session_count":
    daily = user_sessions.groupBy("date").agg(F.count("session_id").alias("y"))
else:
    daily = user_sessions.groupBy("date").agg(F.avg("duration_minutes").alias("y"))

daily_df = daily.orderBy("date").toPandas().rename(columns={"date": "ds"})

In [12]:
daily_df.tail(10)

,ds,y
943,2009-04-09,2
944,2009-04-11,3
945,2009-04-12,2
946,2009-04-16,7
947,2009-04-17,8
948,2009-04-19,4
949,2009-04-20,16
950,2009-04-21,2
951,2009-04-25,86
952,2009-04-28,2


In [13]:
m = Prophet()
m.fit(daily_df)
future = m.make_future_dataframe(periods=FORECAST_HORIZON_DAYS)
predictions = m.predict(future)[["ds", "yhat", "yhat_lower", "yhat_upper"]]

12:09:55 - cmdstanpy - INFO - Chain [1] start processing
12:09:55 - cmdstanpy - INFO - Chain [1] done processing


In [14]:
predictions.tail(10)

,ds,yhat,yhat_lower,yhat_upper
1033,2009-07-18,-21.934393,-174.275596,135.949209
1034,2009-07-19,-8.155497,-160.078197,147.041385
1035,2009-07-20,-0.886773,-159.947592,159.339539
1036,2009-07-21,-9.870366,-166.928089,141.979742
1037,2009-07-22,-20.745971,-176.516510,126.787818
1038,2009-07-23,-12.238527,-159.276873,136.438564
1039,2009-07-24,5.923449,-142.794586,164.781363
1040,2009-07-25,-10.593844,-161.756782,136.374012
1041,2009-07-26,4.271262,-148.531289,150.551009
1042,2009-07-27,12.240263,-136.315619,165.364291


In [15]:
import os
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
predictions.to_csv(OUTPUT_FILE, sep="\t", index=False)
print(f"Forecast written to {OUTPUT_FILE}")

Forecast written to ../data/results/exercise3_forecast.tsv
